# Qwen anchor layer profile map

Per-case layer profile визуализация. Для каждого кейса и каждого anchor profile строятся кривые `r1`, `coherence`, `tortuosity` по всем слоям.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q --upgrade pip
!pip install -q torch torchvision pillow accelerate einops pytest numpy matplotlib nbformat "transformers @ git+https://github.com/huggingface/transformers.git@main"


In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
else:
    %cd /content/ABPT
    !git pull --ff-only

%cd /content/ABPT
print('HEAD =', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen3.5-4B'
DEVICE = 'cuda'
MAX_LENGTH = 160
PROFILES = ['short', 'medium', 'long']
PROFILE_JSON = '/content/ABPT/archive/qwen35_4b_anchor_layer_profile_map.json'
PROFILE_MD = '/content/ABPT/docs/research/qwen35_4b_anchor_layer_profile_map.md'
FIGURE_DIR = '/content/ABPT/docs/research/figures/qwen_anchor_layer_profiles'

for path in [PROFILE_JSON, PROFILE_MD, FIGURE_DIR]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME =', MODEL_NAME)
print('PROFILES =', PROFILES)


In [ ]:
import subprocess
cmd = [
    'python', 'scripts/run_qwen_anchor_layer_profile_map.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--max_length', str(MAX_LENGTH),
    '--profiles', *PROFILES,
    '--output_json', PROFILE_JSON,
    '--output_md', PROFILE_MD,
    '--figure_dir', FIGURE_DIR,
]
print('RUN:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    raise RuntimeError(f'layer profile map failed with code {result.returncode}')


In [ ]:
payload = json.loads(Path(PROFILE_JSON).read_text(encoding='utf-8'))
print('metadata:')
print(json.dumps(payload['metadata'], indent=2, ensure_ascii=False))
print()
print('report head:')
print(Path(PROFILE_MD).read_text(encoding='utf-8')[:7000])
print()
figures = sorted(Path(FIGURE_DIR).glob('*.png'))[:6]
print('sample figures:')
for fig in figures:
    print(fig)
